# 04. Comparing ICL and LoRA: results and errors

Final configurations of `01_icl` and `02_lora` on `rand-test` and `freq-test`: scores, significance of the difference,
which senses each method gets wrong, and whether the lemma helps both methods in the same way.
Runs on CPU from `results/` (no model needed). The detailed ICL error analysis is in `01b_icl_error_analysis`.

In [ ]:
# Setup: on Colab or Kaggle, clone (or update) the repo so that `src/` is importable; locally, run from the repo root.
import os, sys
REPO_URL = "https://github.com/bkadyrbay-hash/lab1-supersense-llm.git"
BRANCH = "b/lora"  # set to your own branch to test code that is not merged yet
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if "google.colab" in sys.modules or ON_KAGGLE:
    REPO_DIR = ("/kaggle/working" if ON_KAGGLE else "/content") + "/lab1-supersense-llm"
    if not os.path.exists(REPO_DIR):
        !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.data import SUPERSENSES, get_split, load_data
from src.evaluate import RESULTS_DIR, evaluate, paired_bootstrap, plot_confusion

FIG_DIR = RESULTS_DIR / "figures"
TEST_SPLITS = ["rand-test", "freq-test"]
METHODS = ["icl", "lora"]
metrics = pd.read_csv(RESULTS_DIR / "metrics.csv").drop_duplicates(["method", "config_id", "split"], keep="last")
train_counts = get_split(load_data(), "train")["supersense"].value_counts()

## 1. Final scores

In [ ]:
final = metrics[metrics["split"].isin(TEST_SPLITS) & metrics["method"].isin(METHODS)]
final_ids = final.groupby(["split", "method"])["config_id"].last()  # the configuration run on each test set
final.set_index(["split", "method"])[["config_id", "macro_f1_super", "accuracy_super", "macro_f1_hyper",
                                      "accuracy_hyper", "invalid_rate"]].sort_index()

## 2. Is the difference consistent?

Paired bootstrap on each test set: share of resamples where LoRA has a higher macro-F1 than ICL.
Close to 100 % or 0 % means a consistent difference, around 50 % means none.

In [ ]:
both = {}
for split in TEST_SPLITS:
    icl, lora = (pd.read_csv(RESULTS_DIR / "predictions" / f"{m}__{final_ids[split, m]}__{split}.csv",
                             keep_default_na=False) for m in METHODS)
    both[split] = icl.merge(lora[["sense_id", "pred_supersense"]], on="sense_id", suffixes=("_icl", "_lora")).assign(
        correct_icl=lambda d: d["pred_supersense_icl"] == d["gold_supersense"],
        correct_lora=lambda d: d["pred_supersense_lora"] == d["gold_supersense"])
    share = paired_bootstrap(both[split]["gold_supersense"], both[split]["pred_supersense_icl"],
                             both[split]["pred_supersense_lora"], SUPERSENSES)
    print(f"{split}: LoRA beats ICL in {share:.0%} of bootstrap resamples")

## 3. Do the methods make the same errors?

In [ ]:
pd.concat({split: pd.crosstab(d["correct_icl"], d["correct_lora"], normalize="all").round(3)
           for split, d in both.items()})

## 4. Per-class F1

In [ ]:
fig, axes = plt.subplots(1, len(TEST_SPLITS), figsize=(11, 7), sharex=True, sharey=True)
per_class = {}
for ax, (split, d) in zip(axes, both.items()):
    per_class[split] = pd.DataFrame({m: evaluate(d["gold_supersense"].tolist(), d[f"pred_supersense_{m}"].tolist(),
                                                 SUPERSENSES)["per_class"] for m in METHODS}).assign(
        n_test=d["gold_supersense"].value_counts(), n_train=train_counts).sort_values("n_train")
    per_class[split][METHODS].plot.barh(ax=ax, width=0.8)
    ax.set_title(f"{split} (classes sorted by train frequency)")
    ax.set_xlabel("F1")
fig.tight_layout()
fig.savefig(FIG_DIR / "per_class_f1_icl_vs_lora.png", dpi=200)
plt.show()
pd.concat(per_class).round(3)

## 5. Confusions of LoRA

ICL confusion matrices are produced by `01b_icl_error_analysis`.

In [ ]:
for split, d in both.items():
    plot_confusion(d["gold_supersense"].tolist(), d["pred_supersense_lora"].tolist(), SUPERSENSES,
                   FIG_DIR / f"lora_confusion_{split}.png")
    errors = d[~d["correct_lora"]]
    print(split, ": most frequent LoRA errors (gold, predicted)")
    display(errors.groupby(["gold_supersense", "pred_supersense_lora"]).size().nlargest(10).to_frame("n"))

## 6. Error rate by property of the input

Same shallow cues as in `00_data_exploration` and `01b_icl_error_analysis`.

In [ ]:
rows = []
for split, d in both.items():
    props = d.assign(
        length=pd.cut(d["definition"].str.split().str.len(), [-1, 3, 7, 15, 1000], labels=["0-3", "4-7", "8-15", ">15"]),
        demonym=d["definition"].str.match(r"Habitant", case=False),
        synonym_or_variant=d["definition"].str.match(r"(Synonyme|Variante|Autre nom|Pluriel) de", case=False),
        complex_type=d["gold_supersense"].str.contains("*", regex=False))
    rows += [(split, prop, value, len(g), 1 - g["correct_icl"].mean(), 1 - g["correct_lora"].mean())
             for prop in ["length", "demonym", "synonym_or_variant", "complex_type"]
             for value, g in props.groupby(prop, observed=True)]
pd.DataFrame(rows, columns=["split", "property", "value", "n", "error_icl", "error_lora"]).round(3)

## 7. Senses that both methods get wrong

In [ ]:
pd.set_option("display.max_colwidth", 90)
hard = pd.concat({split: d[~d["correct_icl"] & ~d["correct_lora"]] for split, d in both.items()})
display(hard.groupby("gold_supersense").size().nlargest(10).to_frame("n_both_wrong"))
hard.sample(min(30, len(hard)), random_state=42)[
    ["lemma", "definition", "gold_supersense", "pred_supersense_icl", "pred_supersense_lora"]]

## 8. Does the lemma help both methods?

Every `rand-dev` run without the lemma, next to the same configuration with the lemma.

In [ ]:
dev_f1 = metrics[metrics["split"] == "rand-dev"].set_index(["method", "config_id"])["macro_f1_super"]
pd.DataFrame([(m, cid.replace("_nolemma", "_lemma"), dev_f1[m, cid.replace("_nolemma", "_lemma")], f1)
              for (m, cid), f1 in dev_f1.items()
              if "_nolemma" in cid and (m, cid.replace("_nolemma", "_lemma")) in dev_f1.index],
             columns=["method", "config_id", "macro_f1_with_lemma", "macro_f1_without_lemma"])